In [0]:
%run /Workspace/Users/ashishbudz@gmail.com/databricks_pipeline/1_setup/utilities


In [0]:
print(bronze_schema)
print(silver_schema)
print(gold_schema)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
#Creating widgets
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

#Retrieving values of widgets
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
print(catalog)
print(data_source)

In [0]:
#Path from where to read the customers data from
base_path = f's3://sportsbar-dp-child-company-prac/{data_source}/*.csv'


In [0]:
#Reading the data from the S3 bucket using the base path
df =( 
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:

display(df)

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .mode("overwrite")\
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')
    

Silver Layer Processing


In [0]:
df_silver = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source};')

In [0]:
display(df_silver)

## Handling Duplicate Values


In [0]:
display(df_silver.groupBy("customer_id").count().filter(F.col("count") > 1))

In [0]:
duplicate_customers = [789321,789503,789522,789603]
display(df_silver.select("*").where(F.col("customer_id").isin(duplicate_customers)).orderBy(F.col("customer_id")))

# Drop the repeating customer ID since the entire rows are duplicates 


In [0]:
df_silver = df_silver.dropDuplicates(["customer_id"]) 


In [0]:
display(df_silver.groupBy("customer_id").count())

## Triming leading spaces


In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
#Triming leading spaaces 
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)



In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

# 
Dealing with distinct City value 

In [0]:
display(
    df_silver.select("city").distinct()
)

In [0]:
#Correct city mapping
city_mapping = {
    'Bengaluruu':'Bengaluru',
    'Bengalore':'Bengaluru',

    'Hyderabadd':'Hyderabad',
    'Hyderbad':'Hyderabad',

    'NewDelhi':'New Delhi',
    'NewDheli':'New Delhi',
    'NewDelhee':'New Delhi'
}

allowed_cities = ['New Delhi','Hyderabad', 'Bengaluru']

In [0]:
df_silver = (
    df_silver.replace(city_mapping, subset=['city'])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
        .when(F.col("city").isin(allowed_cities), F.col("city"))
        .otherwise(None)
    )
)

In [0]:
display(
    df_silver.select("*")
)

# Standardize values : Capitalization, Date format


In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
    .otherwise(F.initcap("customer_name"))
)

In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
display(df_silver.filter(F.col("city").isNull()))

In [0]:
df_silver.groupBy(['customer_name', 'city']).count().show()